# 01 — Data Collection 

## 1. Dependencies

In [ ]:
!pip install -q datasets pandas

### Mount drive & Hugging Face

In [ ]:
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [ ]:
from huggingface_hub import login, HfApi, whoami

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
user_info = whoami()
print(f"Logged in as: {user_info['name']}")

Logged in as: AnoraLee


## 2. ViHSD — Dữ liệu gán nhãn chính

In [ ]:
import pandas as pd
from datasets import load_dataset

LABEL_MAP_VIHSD = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}

def normalize_vihsd(df: pd.DataFrame, split: str) -> pd.DataFrame:
    """Chuẩn hóa ViHSD về schema chung: text, label, source, split."""
    return pd.DataFrame({
        "text": df["free_text"],
        "label": df["label_id"].map(LABEL_MAP_VIHSD),
        "source": "ViHSD",
        "split": split,
    })

vihsd = load_dataset("uitnlp/vihsd")

vihsd_train = normalize_vihsd(vihsd["train"].to_pandas(), "train")
vihsd_dev   = normalize_vihsd(vihsd["validation"].to_pandas(), "dev")
vihsd_test  = normalize_vihsd(vihsd["test"].to_pandas(), "test")

for name, split_df in [("vihsd_train", vihsd_train), ("vihsd_dev", vihsd_dev), ("vihsd_test", vihsd_test)]:
    split_df.to_csv(RAW_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

print(f"ViHSD: train={len(vihsd_train):,} | dev={len(vihsd_dev):,} | test={len(vihsd_test):,}")

train.csv:   0%|          | 0.00/1.62M [00:00<?, ?B/s]

dev.csv:   0%|          | 0.00/177k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24048 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2672 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6680 [00:00<?, ? examples/s]

ViHSD: train=24,048 | dev=2,672 | test=6,680


## 3. VOZ Corpus (Unlabels)


In [ ]:
VOZ_REPO_ID = "AnoraLee/voz-text-corpus-500k"

voz_corpus = load_dataset(VOZ_REPO_ID)["train"].to_pandas()
voz_corpus.to_csv(RAW_DIR / "tdtu_voz_corpus.csv", index=False, encoding="utf-8-sig")

print(f"VOZ corpus: {len(voz_corpus):,} lines")
voz_corpus.head(3)

README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 49.7MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

VOZ corpus: 500,000 lines


,text,source
0,Em ăn hoành thánh sáng bị khó chịu mắc ói quá ...,VOZ-HSD
1,thím chắc khoảng u35 đúng ko ? mấy chuyện này ...,VOZ-HSD
2,Quan trọng là năm nay có tham gia những lễ hội...,VOZ-HSD


## 4. ViHOS — Dữ liệu tham chiếu cấp span (span-level)


In [ ]:
VIHOS_FILES = {
    "train": "https://huggingface.co/datasets/phusroyal/ViHOS/raw/main/train_span_extraction/train.csv",
    "dev":   "https://huggingface.co/datasets/phusroyal/ViHOS/raw/main/train_span_extraction/dev.csv",
    "test":  "https://huggingface.co/datasets/phusroyal/ViHOS/raw/main/test/test.csv",
}

vihos = load_dataset("csv", data_files=VIHOS_FILES)
vihos_all = pd.concat(
    [vihos[split].to_pandas().assign(split=split) for split in VIHOS_FILES],
    ignore_index=True,
)
vihos_all.to_csv(RAW_DIR / "vihos.csv", index=False, encoding="utf-8-sig")

print(f"ViHOS: {len(vihos_all):,} lines (train / dev / test)")

ViHOS: 11,056 lines (train / dev / test)


## 5. Kiểm tra nhanh trước khi bàn giao (Sanity Check)


In [ ]:
for fname in ["vihsd_train.csv", "vihsd_dev.csv", "vihsd_test.csv", "tdtu_voz_corpus.csv", "vihos.csv"]:
    path = RAW_DIR / fname
    df = pd.read_csv(path)
    print(f"{fname:<22} shape={df.shape}  nulls={df.isna().sum().sum()}")

vihsd_train.csv        shape=(24048, 4)  nulls=2
vihsd_dev.csv          shape=(2672, 4)  nulls=0
vihsd_test.csv         shape=(6680, 4)  nulls=0
tdtu_voz_corpus.csv    shape=(640, 10)  nulls=641
vihos.csv              shape=(11056, 4)  nulls=0


## 6. Thống kê data HuggingFace


In [ ]:
summary = []
for name in ["vihsd_train", "vihsd_dev", "vihsd_test", "tdtu_voz_corpus", "vihos"]:
    df = pd.read_csv(RAW_DIR / f"{name}.csv")
    row = {"file": name, "rows": len(df), "cols": df.shape[1]}
    if "label" in df.columns:
        row["label_dist"] = df["label"].value_counts().to_dict()
    summary.append(row)

for row in summary:
    print(f"{row['file']:<18} rows={row['rows']:>8,}  cols={row['cols']}", end="")
    print(f"  labels={row['label_dist']}" if "label_dist" in row else "")

print(f"\nTotal lines: {sum(r['rows'] for r in summary):,}")

vihsd_train        rows=  24,048  cols=4  labels={'CLEAN': 19886, 'HATE': 2556, 'OFFENSIVE': 1606}
vihsd_dev          rows=   2,672  cols=4  labels={'CLEAN': 2190, 'HATE': 270, 'OFFENSIVE': 212}
vihsd_test         rows=   6,680  cols=4  labels={'CLEAN': 5548, 'HATE': 688, 'OFFENSIVE': 444}
tdtu_voz_corpus    rows= 500,000  cols=2
vihos              rows=  11,056  cols=4

Total lines: 544,456
